# ASR A/B preference rating (Colab / Workbench)

1. Set **RATER_ID** (and optionally **REDO**) in cell 1
2. Run **Setup** cell once (git pull + pip install)
3. Run **Launch** cell
4. Rate **Text / Timing / Diarization** (A / B / Tie each), then **Next item**

Already-rated items are skipped automatically. Set `REDO = True` to wipe your ratings and start from scratch.

On Colab the rating UI uses HTML + kernel callbacks (not ipywidgets — Colab pre-loads ipywidgets before any cell runs).

See [COLAB.md](COLAB.md) for lead setup.

---

### ⌨️ Keyboard shortcuts

| Key | Action |
|-----|--------|
| `Space` | Play / pause audio |
| `1` `2` `3` | Text → A / B / Tie |
| `q` `w` `e` | Timing → A / B / Tie |
| `a` `s` `d` | Diarization → A / B / Tie |
| `Enter` | Submit (Next item) — only when all 3 rows are picked |
| `n` | Toggle Add note field |

In [ ]:
# --- edit these ---
RATER_ID = "Alice"
REDO = False  # True = WIPE your existing ratings and start from scratch (use with caution!)

SESSION_CONFIG = "configs/ab_prefs.session.multi_dimension.v3.json"  # v3: aai_up3_v2 vs qwen_3_1_7B vs qwen_3_0_6B
RUNTIME_SESSION_CONFIG = "configs/ab_prefs.session.runtime.multi_dimension.v3.json"  # bootstrap writes GCS paths here
GCS_BUCKET = "dd_tfx_full_transcripts"
MOUNT_POINT = "/content/dd_tfx"  # Workbench: e.g. "/home/jupyter/gcs/dd_tfx_full_transcripts"
REPO_ROOT = "/content/ab_prefs_rating"
REPO_URL = "https://github.com/rosyvs/ab_prefs_rating.git"
ASR_SUBDIR = "asr/dd210"

In [ ]:
# --- Setup: git pull + pip install (run once per session, or after pulling code changes) ---
import subprocess, sys
from pathlib import Path

repo = Path(REPO_ROOT)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
elif not (repo / "pyproject.toml").is_file():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)], check=True)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

# Show commit so you can verify Colab has the latest version
commit_id = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "--short", "HEAD"], text=True).strip()
commit_date = subprocess.check_output(["git", "-C", str(repo), "log", "-1", "--format=%ci"], text=True).strip()
commit_msg = subprocess.check_output(["git", "-C", str(repo), "log", "-1", "--format=%s"], text=True).strip()
print(f"✓ ab_prefs_rating @ {commit_id}  ({commit_date})")
print(f"  {commit_msg}")

In [ ]:
# --- Launch rating UI ---
import sys
from pathlib import Path

repo = Path(REPO_ROOT)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from ab_prefs_interface.colab_setup import bootstrap
from ab_prefs_interface.launch import launch_rating

repo, session_config = bootstrap(
    RATER_ID,
    bucket=GCS_BUCKET,
    mount_point=MOUNT_POINT,
    repo_root=repo,
    repo_url=REPO_URL,
    asr_subdir=ASR_SUBDIR,
    session_config=SESSION_CONFIG,
    runtime_session_config=RUNTIME_SESSION_CONFIG,
)
_ = launch_rating(RATER_ID, session_config_path=session_config, redo=REDO, gcs_bucket=GCS_BUCKET)

In [ ]:
# Optional: upload ratings to GCS
# !gsutil cp {REPO_ROOT}/results/ab_prefs_multi_dimension/preferences_{RATER_ID}.json gs://{GCS_BUCKET}/ab_prefs_multi_dimension/